# 🐛 Bug Explainer — QLoRA Finetuning
### Built for Kaggle CUDA 13.0 environment

> ⚡ Before starting:
> - Settings → Accelerator → **GPU T4 x2**
> - Settings → Internet → **ON**
> - Run cells **one by one** — don't Run All

## 📦 CELL 1 — Install (run alone, then restart kernel)

In [1]:
import subprocess, sys

# Step 1: Upgrade numpy first (needs 2.0+ for modern transformers)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=2.0'
], check=False)

# Step 2: Uninstall conflicting packages
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'transformers', 'peft', 'trl', 'accelerate',
    'tokenizers', 'bitsandbytes', 'datasets'
], check=False)

# Step 3: Install compatible stack for CUDA 13 / PyTorch 2.11
packages = [
    'transformers==4.46.0',
    'peft==0.13.2',
    'trl==0.11.4',
    'accelerate==1.0.1',
    'tokenizers==0.20.3',
    'datasets==3.0.1',
    'huggingface_hub',
    'requests',
    'beautifulsoup4',
    'tqdm',
    'evaluate',
    'rouge_score',
    'gradio',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    check=False
)


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2


Found existing installation: datasets 4.8.3
Uninstalling datasets-4.8.3:
  Successfully uninstalled datasets-4.8.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.3 MB/s eta 0:00:00


Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.2 MB/s eta 0:00:00
✅ Installation complete!

🔴 IMPORTANT — Do this before running Cell 2:
   Run menu → Restart & Clear Output
   Then run

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.6.1 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


## 🔧 CELL 2 — Patch broken imports (CUDA 13 compatibility)

## 📚 CELL 3 — Imports & GPU check

In [3]:
import os, json, time
import torch
import requests
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from huggingface_hub import login, HfApi

import transformers, peft, trl, accelerate

    

2026-05-10 11:25:43.183516: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778412343.671572      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778412343.779724      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778412344.822884      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778412344.822927      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778412344.822929      57 computation_placer.cc:177] computation placer alr

Library versions
  numpy        : 2.0.2
  torch        : 2.10.0+cu128
  transformers : 4.46.0
  peft         : 0.13.2
  trl          : 0.11.4
  accelerate   : 1.0.1

GPU Status
  CUDA available : True
  CUDA version   : 12.8
  GPU 0          : Tesla T4 (15.6 GB)
  GPU 1          : Tesla T4 (15.6 GB)

✅ All imports successful — ready to train!


## 🌐 CELL 4 — Build Dataset (Stack Overflow + handcrafted)

In [4]:
import re
from bs4 import BeautifulSoup
from collections import Counter

SO_API       = 'https://api.stackexchange.com/2.3'
MAX_PER_LANG = 100    # increase for more data
DELAY        = 0.5

LANG_CONFIG = {
    'Python':     {'tags': 'python',     'keywords': ['Error', 'Exception', 'Traceback']},
    'JavaScript': {'tags': 'javascript', 'keywords': ['TypeError', 'ReferenceError', 'SyntaxError']},
    'Java':       {'tags': 'java',       'keywords': ['Exception', 'NullPointerException']},
    'C++':        {'tags': 'c++',        'keywords': ['error:', 'segmentation fault']},
}

def clean_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    for code in soup.find_all('code'):
        code.replace_with(f'`{code.get_text()}`')
    text = soup.get_text(separator=' ')
    return re.sub(r'\s+', ' ', text).strip()

def fetch_questions(tag, keywords):
    questions, page = [], 1
    while len(questions) < MAX_PER_LANG:
        try:
            r = requests.get(f'{SO_API}/questions', params={
                'order': 'desc', 'sort': 'votes',
                'tagged': tag, 'site': 'stackoverflow',
                'filter': 'withbody', 'pagesize': 50,
                'page': page, 'has_accepted_answer': True,
            }, timeout=10)
            data = r.json()
        except Exception as e:
            print(f'API error: {e}')
            break
        if 'items' not in data:
            break
        for q in data['items']:
            title = q.get('title', '')
            body  = clean_html(q.get('body', ''))[:600]
            if any(kw.lower() in (title + body).lower() for kw in keywords):
                questions.append({
                    'title': title, 'body': body,
                    'accepted_id': q.get('accepted_answer_id'),
                })
            if len(questions) >= MAX_PER_LANG:
                break
        if not data.get('has_more', False):
            break
        page += 1
        time.sleep(DELAY)
    return questions

def fetch_answer(answer_id):
    try:
        r = requests.get(f'{SO_API}/answers/{answer_id}',
            params={'site': 'stackoverflow', 'filter': 'withbody'}, timeout=10)
        items = r.json().get('items', [])
        if items:
            return clean_html(items[0].get('body', ''))[:600]
    except:
        pass
    return ''

# ── Scrape ───────────────────────────────────────────────────
scraped = []
for lang, cfg in LANG_CONFIG.items():
    print(f'Scraping {lang}...')
    qs = fetch_questions(cfg['tags'], cfg['keywords'])
    for q in tqdm(qs, desc=f'  {lang}'):
        if not q['accepted_id']:
            continue
        ans = fetch_answer(q['accepted_id'])
        if len(ans) < 50:
            continue
        scraped.append({
            'language': lang,
            'error': f"{q['title']}\n\n{q['body']}",
            'explanation': ans,
        })
        time.sleep(DELAY)

# ── Handcrafted high-quality examples ────────────────────────
handcrafted = [
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "app.py", line 12\n    result = my_list[10]\nIndexError: list index out of range',
        'explanation': 'You are trying to access index 10 of a list that does not have that many elements. Python lists are zero-indexed, so a list with 5 items has indices 0 to 4. Check the length first with len(my_list), or use a try/except IndexError block.',
    },
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "main.py", line 5\n    print(name)\nNameError: name "name" is not defined',
        'explanation': 'You are using the variable name before defining it. Python requires variables to be assigned before use. Define it first: name = "Alice", then use it.',
    },
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "calc.py", line 8\n    return a / b\nZeroDivisionError: division by zero',
        'explanation': 'Your code is dividing by zero. The variable b equals 0 at this point. Add a check before dividing: if b != 0: return a / b, otherwise return a default value or raise a meaningful error.',
    },
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "data.py", line 15\n    value = data["username"]\nKeyError: "username"',
        'explanation': 'You are accessing a key that does not exist in the dictionary. Either it is misspelled or was never set. Use data.get("username") to safely return None if missing, or check with: if "username" in data first.',
    },
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "model.py", line 22\n    result = obj.predict(data)\nAttributeError: NoneType object has no attribute predict',
        'explanation': 'The variable obj is None instead of the object you expected. A function returned None, possibly due to a failed initialization. Add a check: if obj is not None: before calling methods on it, and trace back where obj was assigned.',
    },
    {
        'language': 'JavaScript',
        'error': 'TypeError: Cannot read properties of undefined (reading "map")\n    at App.js:15:23',
        'explanation': 'You are calling .map() on a variable that is undefined. This usually means an API response has not loaded yet. Add a check before mapping: if (data && Array.isArray(data)) { data.map(...) }, or use optional chaining: data?.map(...).',
    },
    {
        'language': 'JavaScript',
        'error': 'ReferenceError: fetch is not defined\n    at getUser (api.js:12:18)',
        'explanation': 'The fetch API is not available in your environment. This happens in older Node.js versions (below 18) or certain server environments. Install node-fetch: npm install node-fetch, then import it: const fetch = require("node-fetch").',
    },
    {
        'language': 'Java',
        'error': 'Exception in thread "main" java.lang.NullPointerException\n\tat com.example.Main.processUser(Main.java:42)',
        'explanation': 'A NullPointerException means you are calling a method or accessing a field on an object that is null. At line 42, one of your variables was never initialized or a method returned null. Add a null check: if (obj != null) before using it.',
    },
    {
        'language': 'Java',
        'error': 'Exception in thread "main" java.util.ConcurrentModificationException\n\tat java.util.ArrayList$Itr.next(ArrayList.java:967)',
        'explanation': 'You are modifying a list while iterating over it with a for-each loop, which is not allowed. Use an Iterator with iterator.remove() instead, or collect items to remove in a separate list and remove them after the loop.',
    },
    {
        'language': 'C++',
        'error': 'Segmentation fault (core dumped)\n#0 in processData (ptr=0x0) at main.cpp:28',
        'explanation': 'A segmentation fault means your program accessed invalid memory. The pointer ptr is null (0x0) at line 28. Always check pointers before dereferencing: if (ptr != nullptr) { ... }. Also check for array out-of-bounds access nearby.',
    },
    {
        'language': 'C++',
        'error': 'error: use of deleted function std::unique_ptr<int>::unique_ptr(const std::unique_ptr<int>&)\n  MyClass obj2 = obj1;',
        'explanation': 'unique_ptr cannot be copied because it has exclusive ownership of its resource. You are trying to copy obj1 which contains a unique_ptr. Use std::move() to transfer ownership: MyClass obj2 = std::move(obj1), or use shared_ptr if you need shared ownership.',
    },
    {
        'language': 'Python',
        'error': 'Traceback (most recent call last):\n  File "async_app.py", line 10\n    result = my_coroutine()\nRuntimeError: coroutine was never awaited',
        'explanation': 'You called an async function without the await keyword. Without await, Python just creates a coroutine object but never runs it. Fix: result = await my_coroutine(), or if outside async context use: result = asyncio.run(my_coroutine()).',
    },
]

all_samples = handcrafted + scraped
print(f'\n✅ Total samples: {len(all_samples)}')
print('Breakdown by language:')
for lang, count in Counter(s['language'] for s in all_samples).items():
    print(f'  {lang}: {count}')

# Save to disk
with open('dataset.json', 'w') as f:
    json.dump(all_samples, f, indent=2)
print('\n💾 Saved to dataset.json')

Scraping Python...


  Python: 100%|██████████| 100/100 [00:50<00:00,  1.97it/s]


Scraping JavaScript...


  JavaScript: 100%|██████████| 15/15 [00:07<00:00,  2.09it/s]


Scraping Java...


  Java: 100%|██████████| 100/100 [00:56<00:00,  1.77it/s]


Scraping C++...


  C++: 100%|██████████| 62/62 [00:17<00:00,  3.55it/s]


✅ Total samples: 228
Breakdown by language:
  Python: 90
  JavaScript: 14
  Java: 96
  C++: 28

💾 Saved to dataset.json


## 🔧 CELL 5 — Format as instruction pairs

In [ ]:
def format_sample(s):
    lang = s.get('language', 'Python')
    return {
        'text': (
            f'### Instruction:\n'
            f'You are an expert {lang} debugging assistant. '
            f'Explain the following error in plain English and give a clear fix.\n\n'
            f'### Language: {lang}\n\n'
            f'### Error:\n{s["error"]}\n\n'
            f'### Explanation:\n{s["explanation"]}'
        )
    }

formatted    = [format_sample(s) for s in all_samples]
hf_dataset   = Dataset.from_list(formatted)
split        = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_data   = split['train']
eval_data    = split['test']

print(f'✅ Train : {len(train_data)} samples')
print(f'✅ Eval  : {len(eval_data)} samples')
print()
print('Sample prompt preview:')
print('-' * 50)
print(train_data[0]['text'][:500])
print('...')

## 🤖 CELL 6 — Load Mistral-7B in fp16 (no bitsandbytes needed)

We use **fp16** instead of 4-bit because bitsandbytes does not support CUDA 13.0 yet.
Mistral-7B in fp16 uses ~14GB — fits perfectly in T4 x2 (2 × 15.8GB = 31.6GB total).

In [ ]:
from huggingface_hub import login

# ── Login to Hugging Face ────────────────────────────────────
# Get token: https://huggingface.co/settings/tokens → New token → Read
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # ← paste your token here
login(token=HF_TOKEN)

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'

print(f'📥 Loading {MODEL_NAME}')
print('   Downloading ~14GB in fp16 — takes 5-10 mins first run...')
print('   (cached on rerun — will be fast)')

# Load in fp16 — no bitsandbytes needed, works on CUDA 13
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',          # auto-split across both T4 GPUs
    torch_dtype=torch.float16,  # fp16 = ~14GB, fits in T4 x2
    token=HF_TOKEN,
    low_cpu_mem_usage=True,     # reduces RAM during loading
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

model.config.use_cache      = False
model.config.pretraining_tp = 1

print(f'\n✅ Model loaded!')
print(f'   Parameters : {model.num_parameters()/1e9:.2f}B')
print(f'   GPU 0 mem  : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')
if torch.cuda.device_count() > 1:
    print(f'   GPU 1 mem  : {torch.cuda.memory_allocated(1)/1e9:.2f} GB')

## 🔩 CELL 7 — Attach LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=16,                  # LoRA rank
    lora_alpha=32,         # scaling = alpha/r = 2.0
    target_modules=[       # attention + MLP layers
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Critical fix: allows gradients to flow through frozen fp16 layers
model.enable_input_require_grads()

model.print_trainable_parameters()
print('\n✅ LoRA adapters attached!')
print('   Only the tiny LoRA matrices will be trained.')

## ⚙️ CELL 8 — Training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./bug_explainer_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    gradient_checkpointing=False,    # off — causes issues with LoRA + fp16
    optim='adamw_torch',             # standard optimizer (no paged — CUDA 13)
    learning_rate=2e-4,
    weight_decay=0.001,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=5,
    evaluation_strategy='steps',
    eval_steps=25,
    save_strategy='steps',
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,                       # use fp16 for training
    bf16=False,
    max_grad_norm=0.3,
    group_by_length=True,
    report_to='none',
)

print('✅ Training arguments configured!')
eff_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f'   Effective batch size : {eff_batch}')
print(f'   Learning rate        : {training_args.learning_rate}')
print(f'   Epochs               : {training_args.num_train_epochs}')

## 🚀 CELL 9 — Train!

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=lora_config,
    dataset_text_field='text',
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

torch.cuda.empty_cache()

print('🚀 Training started!')
print('   Watch eval_loss — lower is better')
print('   Expected: starts ~2.0, drops to ~0.5 by end\n')

start = time.time()
trainer.train()
elapsed = time.time() - start

print(f'\n🎉 Training complete!')
print(f'   Time taken : {elapsed/60:.1f} minutes')

## 💾 CELL 10 — Save model

In [ ]:
SAVE_PATH = './bug_explainer_adapters'

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

total_mb = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH)
) / 1e6

metrics = trainer.evaluate()

print(f'✅ Saved to: {SAVE_PATH}')
print(f'   Adapter size  : {total_mb:.1f} MB')
print(f'   Files         : {os.listdir(SAVE_PATH)}')
print(f'   Final eval loss : {metrics["eval_loss"]:.4f}')

## 🤗 CELL 11 — Push to Hugging Face Hub

In [ ]:
HF_USERNAME = 'your-username'          # ← your HF username
REPO_NAME   = 'bug-explainer-mistral'  # ← repo name
REPO_ID     = f'{HF_USERNAME}/{REPO_NAME}'

# Push model + tokenizer
trainer.model.push_to_hub(REPO_ID, token=HF_TOKEN,
    commit_message='Upload Bug Explainer LoRA adapters')
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)

# Auto-generate model card
api = HfApi()
card = f"""---
language: en
license: apache-2.0
base_model: mistralai/Mistral-7B-Instruct-v0.2
tags:
- debugging
- code
- lora
- peft
---
# 🐛 Bug Explainer — Mistral-7B LoRA
Finetuned from Mistral-7B-Instruct-v0.2 on Stack Overflow error/answer pairs.
Supports Python, JavaScript, Java, C++.
"""
api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo='README.md',
    repo_id=REPO_ID,
    token=HF_TOKEN,
)
print(f'\n✅ Model live at: https://huggingface.co/{REPO_ID}')

## 🧪 CELL 12 — Test the model on all 4 languages

In [ ]:
def explain_bug(error, language='Python', max_new_tokens=300):
    prompt = (
        f'### Instruction:\n'
        f'You are an expert {language} debugging assistant. '
        f'Explain the following error in plain English and give a clear fix.\n\n'
        f'### Language: {language}\n\n'
        f'### Error:\n{error}\n\n'
        f'### Explanation:\n'
    )
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=512
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full.split('### Explanation:')[-1].strip()


test_cases = [
    ('Python',
     'Traceback (most recent call last):\n  File "app.py", line 8\n    result = data["user"]["profile"]["name"]\nKeyError: "profile"'),
    ('JavaScript',
     'Uncaught TypeError: data.filter is not a function\n    at processResults (app.js:45:18)'),
    ('Java',
     'Exception in thread "main" java.lang.StackOverflowError\n\tat Fibonacci.fib(Fibonacci.java:8)'),
    ('C++',
     'undefined reference to `MyClass::myMethod()\ncollect2: error: ld returned 1 exit status'),
]

for lang, error in test_cases:
    print('=' * 60)
    print(f'Language : {lang}')
    print(f'Error    : {error[:80]}...')
    print(f'\nExplanation:')
    print(explain_bug(error, lang))
    print()

## 📊 CELL 13 — Evaluate with ROUGE score

In [ ]:
import evaluate

rouge = evaluate.load('rouge')

eval_samples = all_samples[-15:]
preds, refs  = [], []

print('Running evaluation...')
for s in tqdm(eval_samples):
    pred = explain_bug(s['error'], s['language'])
    preds.append(pred)
    refs.append(s['explanation'])

results = rouge.compute(predictions=preds, references=refs)

print('\n📊 ROUGE Results:')
print(f'   ROUGE-1 : {results["rouge1"]:.4f}  (word overlap)')
print(f'   ROUGE-2 : {results["rouge2"]:.4f}  (bigram overlap)')
print(f'   ROUGE-L : {results["rougeL"]:.4f}  (sequence match)')
print()
print('Benchmarks:')
print('   ROUGE-L > 0.3 = decent')
print('   ROUGE-L > 0.5 = very good')

## 🎨 CELL 14 — Gradio Web UI

In [ ]:
import gradio as gr

EXAMPLES = [
    ['Python',     'Traceback (most recent call last):\n  File "app.py", line 5\n    print(my_list[99])\nIndexError: list index out of range'],
    ['JavaScript', 'TypeError: Cannot read properties of null (reading "addEventListener")\n    at main.js:23:45'],
    ['Java',       'Exception in thread "main" java.lang.NullPointerException\n\tat com.example.Main.process(Main.java:42)'],
    ['C++',        'error: \'vector\' was not declared in this scope\n  vector<int> nums = {1, 2, 3};'],
]

def run(language, error):
    if not error.strip():
        return '⚠️ Please paste an error above.'
    try:
        return explain_bug(error, language)
    except Exception as e:
        return f'Error: {e}'

with gr.Blocks(title='Bug Explainer', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🐛 Bug Explainer\nPaste any error → get a plain English explanation + fix')

    with gr.Row():
        lang_dd = gr.Dropdown(
            ['Python', 'JavaScript', 'Java', 'C++'],
            value='Python', label='Language'
        )
        error_box = gr.Textbox(
            label='Paste your error / traceback',
            placeholder='Traceback (most recent call last):\n  ...',
            lines=8
        )

    btn = gr.Button('🔍 Explain this bug', variant='primary')
    out = gr.Textbox(label='Plain English Explanation + Fix', lines=8, interactive=False)

    gr.Examples(EXAMPLES, inputs=[lang_dd, error_box], label='Try an example')
    btn.click(fn=run, inputs=[lang_dd, error_box], outputs=out)

demo.launch(share=True, debug=True)